In [172]:
%pip install pandas transformers deepparse huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [173]:
import torch
if torch.cuda.is_available():
    print("CUDA - available devices:")
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")
    device = torch.device('cuda')
elif torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    print("WARNING: Running on CPU")
    device = torch.device('cpu')
print(f"Torch version: {torch.__version__}, Device: {device}")

Torch version: 2.10.0+xpu, Device: xpu


In [174]:
from huggingface_hub import notebook_login
notebook_login()

In [175]:
import pandas as pd
from collections import OrderedDict

COLS_TO_PREDICT = [
    "HouseNumber",
    "StreetName",
    "City",
    "State",
    "Country"
]

EXCEPT_COUNTRY = COLS_TO_PREDICT[:-1]

# Adapted Mahsa's code for levenshtein distance
def levenshtein(a: str, b: str, case_insensitive=True) -> int:
    # If one of the strings is empty
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)
    
    if case_insensitive:
        a = a.lower()
        b = b.lower()

    # Create distance matrix (size: (len(a)+1) x (len(b)+1))
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]

    # Initialize first row/column
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j

    # Fill in matrix
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost # substitution
            )
    distance = dp[-1][-1]

    return distance

def compare_preds(preds : pd.DataFrame, labels : pd.DataFrame, target_columns = COLS_TO_PREDICT, ignore_trash_columns = True):
    # Drop meta columns that may be included in the preds dataframe
    labels = labels.astype(str)

    tolerance_levels = 5
    correct_with_tol = [0,] * tolerance_levels
    total_rows = 0
    prediction_count = 0
    label_count = 0
    true_positives = 0
    
    sum_levenshtein = 0
    sum_similarity = 0.0
    sum_levenshtein_match = 0
    sum_similarity_match = 0.0
    some_match_count = 0
    
    if not ignore_trash_columns:
        # labels that should not have been predicted at all
        trash_predictions = preds[[col for col in preds.columns if col not in labels.columns]].stack()
        trash_count = trash_predictions.notna().sum()
        total_rows += trash_count
        prediction_count += trash_count
        sum_levenshtein += trash_predictions.dropna().astype(str).str.len().sum()
    for col in target_columns:
        total_rows += len(labels)
        label_count += labels[col].notna().sum()
        if col not in preds.columns:
            # all missing predictions are incorrect
            sum_levenshtein += labels[col].dropna().str.len().sum()
        else:
            prediction_count += preds[col].notna().sum()
            strings_to_compare = pd.concat([preds[col].fillna(""), labels[col].fillna("")], axis=1)
            levenshtein_scores = strings_to_compare.apply(
                lambda row: levenshtein(row.iloc[0], row.iloc[1]), axis=1
            )
            levenshtein_bounds = strings_to_compare.apply(
                lambda row: max(len(row.iloc[0]), len(row.iloc[1])), axis=1
            )
            similarity = ((levenshtein_bounds - levenshtein_scores) / levenshtein_bounds).fillna(1.0) # nan => div by 0 => both are empty strings => similarity 1.0
            sum_levenshtein += levenshtein_scores.sum()
            sum_similarity += similarity.sum()
            sum_levenshtein_match += levenshtein_scores[similarity >= 0].sum()
            sum_similarity_match += similarity[similarity >= 0].sum()
            some_match_count += (similarity > 0).sum()
            true_positives += ((levenshtein_scores == 0) & preds[col].notna() & labels[col].notna()).sum()
            for tol in range(tolerance_levels):
                correct_with_tol[tol] += (levenshtein_scores <= tol).sum()
    results = OrderedDict()
    results["accuracy"] = correct_with_tol[0] / total_rows
    results["precision"] = true_positives / prediction_count if prediction_count > 0 else 0.0
    results["recall"] = true_positives / label_count if label_count > 0 else 0.0
    results["f1"] = 2 * results["precision"] * results["recall"] / (results["precision"] + results["recall"]) if (results["precision"] + results["recall"]) > 0 else 0.0
    for tol in range(1, tolerance_levels):
        results[f"accuracy_with_tol_{tol}"] = correct_with_tol[tol] / total_rows
    results["average_levenshtein"] = sum_levenshtein / total_rows
    results["average_similarity"] = sum_similarity / total_rows
    results["average_levenshtein_match"] = sum_levenshtein_match / some_match_count if some_match_count > 0 else 0.0
    results["average_similarity_match"] = sum_similarity_match / some_match_count if some_match_count > 0 else 0.0
    results["no_match_rate"] = 1.0 - (some_match_count / total_rows)
    return results

In [176]:


class ParsedAddressResultBuilder:
    def __init__(self, original_address):
        self.inner = {}
        self.original_address = original_address

    def _merge_components(self, conflict : str, new_component : str, separator = "___") -> str:
        if not conflict:
            return new_component
        
         # Ignore repeating matches
        if conflict == new_component:
            return conflict
        
         # Take larger match if one is contained in the other
        if conflict in new_component:
            return new_component 
        if new_component in conflict:
            return conflict
        
        # Try to merge if both are consecutive
        conflict_start = self.original_address.find(conflict[0])
        new_component_start = self.original_address.find(new_component)
        if conflict_start != -1 and new_component_start != -1:
            if conflict_start < new_component_start:
                start = conflict_start
                between = self.original_address[conflict_start + len(conflict[0]) : new_component_start]
                end = new_component_start + len(new_component)
            else:
                start = new_component_start
                between = self.original_address[new_component_start + len(new_component) : conflict_start]
                end = conflict_start + len(conflict[0])
            ignored_chars = set(",. -/\\ \t") # set of separator characters that can be ignored when checking for consecutivity
            if all(x in ignored_chars for x in between):
                return self.original_address[start:end]
            
        # Failure to solve conflict; set prediction to a combination of both components to at least not lose the information
        return conflict + separator + new_component

    
    def add_part(self, label, part):
        if not part: # ignore None or empty
            return
        part = str(part)
        if label in ["fullResponse", "model-fullAddress", "error"]:
            print(f"ERROR: Attempt to add reserved label: {label} for address: {self.original_address}")
            return
        if label == "fullAddress":
            label = "model-fullAddress" # avoid collision but keep the wrongfully generated field
        conflict = self.inner.get(label, None)
        self.inner[label] = self._merge_components(conflict, part)

    def set_reserved(self, label, value):
        self.inner[label] = value

    def build(self) -> dict:
        return self.inner

    


In [177]:
import deepparse.parser

addr_parser = deepparse.parser.AddressParser(device=torch.get_default_device())

print(addr_parser("123 Main Street, Los Angeles, CA, USA").to_dict())
del addr_parser # release gpu resources

c:\Users\rpa\Documents\wiedergutmachung\bzk-post-processing\.conda\Lib\site-packages\deepparse\download_tools.py:92: UserWarning: The offline parameter is set to False, so if a new pre-trained `bpemb` model is available it will automatically be downloaded.
  warnings.warn(


Loading the embeddings model
{'StreetNumber': '123', 'StreetName': 'main street los angeles', 'Unit': None, 'Municipality': 'ca', 'Province': 'usa', 'PostalCode': None, 'Orientation': None, 'GeneralDelivery': None, 'EOS': None}


In [178]:

DEEPPARSE_LABEL_MAPPING = {
    "StreetNumber": "HouseNumber",
    # "StreetName": "StreetName",
    "Municipality": "City",
    #"Province": "State", # note: DeepParse does not distinguish between state, province, region or country
    "Province": "Country",
    "PostalCode" : "PostalCode"
}

class DeepParseParser:
    def __init__(self, **kwargs):
        self.parser = deepparse.parser.AddressParser(device=device, **kwargs)

    def _transform_results(self, parsed_addresses, addresses):
        results = []
        for parsed, addr in zip(parsed_addresses, addresses):
            result_builder = ParsedAddressResultBuilder(addr)
            tuples = parsed.to_list_of_tuples()
            for part, label in tuples:
                result_builder.add_part(DEEPPARSE_LABEL_MAPPING.get(label, label), part)
            results.append(result_builder.build())
        return results

    
    def parse_addresses(self, addresses):
        parsed_results = self.parser(addresses)
        return self._transform_results(parsed_results, addresses)
    


In [179]:
import http.client
import json
import urllib.parse
import time

LIBPOSTAL_LABEL_MAPPING = {
    "house_number": "HouseNumber",
    "road": "StreetName",
    "city": "City",
    "state": "State",
    "country": "Country",
    "postcode": "PostalCode"
}

class LibpostalClient:
    def __init__(self, url: str = "http://localhost:7272", 
                 label_mapping = LIBPOSTAL_LABEL_MAPPING, expand_before_parsing = False,
                 start_container_if_unavailable: bool = True):
        self.url = url
        parsed_url = urllib.parse.urlparse(url)
        self.host = parsed_url.hostname
        self.port = parsed_url.port
        self.label_mapping = label_mapping or {}
        self.expand_before_parsing = expand_before_parsing
        self.auto_started = False
        if start_container_if_unavailable:
            if not self.start_container_if_needed():
                raise ConnectionError(f"Could not connect to libpostal server at {self.url}, and failed to start docker container.")
    
    def _transform_results(self, parsed_addresses : list[list[list[str]]], addresses: list[str]) -> list[dict]:
        results = []
        for parsed, addr in zip(parsed_addresses, addresses):
            result_builder = ParsedAddressResultBuilder(addr)
            for part, label in parsed:
                if label in self.label_mapping:
                    label = self.label_mapping[label]
                result_builder.add_part(label, part)
            results.append(result_builder.build())
        return results
    
    def _handle_request(self, addresses):
        conn = http.client.HTTPConnection(self.host, self.port, timeout=3)
        headers = {'Content-Type': 'application/json'}
        if isinstance(addresses, str):
            addresses = [addresses]
        elif not isinstance(addresses, list):
            addresses = list(addresses)
        body = json.dumps(addresses)
        path = "/parse_addresses"
        if self.expand_before_parsing:
            path += "?expandFirst=true"
        conn.request("GET", path, body, headers)
        response = conn.getresponse()
        data = response.read()
        
        results = json.loads(data.decode('utf-8'))
        conn.close()

        return self._transform_results(results, addresses)
    
    def _health_check(self) -> bool:
        conn = http.client.HTTPConnection(self.host, self.port, timeout=3)
        try:
            conn.request("GET", "/health")
            response = conn.getresponse()
            conn.close()
            return response.status == 204
        except Exception:
            return False
    
    def _start_docker_container(self) -> bool:
        print("Attempting to start libpostal-server docker-compose service...")
        print("This may take a long time on first run since the docker image needs to be built.")
        import subprocess
        result = subprocess.run(["docker-compose", "-f", "docker-compose.yml", "up", "-d", "libpostal-server"], capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Failed to start libpostal-server docker container (exit code {result.returncode}):")
            print(result.stdout)
            print(result.stderr)
            return False
        for _ in range(10):
                time.sleep(1)
                if self._health_check():
                    print("Libpostal server is now available.")
                    self.auto_started = True
                    return True
        return False
    
    def start_container_if_needed(self):
        if not self._health_check():
            return self._start_docker_container()
        else: return True

    def parse_addresses(self, addresses: list[str]):
        try:
            return self._handle_request(addresses)
        except Exception as e:
            if self._health_check():
                raise e
            else:
                print(f"Libpostal server not reachable at {self.url}.")
                if not self._start_docker_container():
                    raise e
            return self._handle_request(addresses)
    
    def close(self):
        if self.auto_started:
            print("Stopping auto-started libpostal-server docker container...")
            import subprocess
            subprocess.run(["docker-compose", "-f", "docker-compose.yml", "down", "libpostal-server"])

In [180]:
import transformers
from transformers import pipeline

class LlamaAddressParsingModel:
    def __init__(self, model_name, prompt, output_parser = None, batch_size=32):
        tokenizer = transformers.AutoTokenizer.from_pretrained(
            "meta-llama/Llama-3.2-1B-Instruct", padding_side='left', device=device)
        self.pipe = pipeline("text-generation", model=model_name, batch_size=batch_size, tokenizer=tokenizer)
        self.pipe.tokenizer.pad_token_id = self.pipe.model.config.eos_token_id[0]
        self.prompt = prompt
        self.output_parser = output_parser or (lambda x, original_address : x)

    def parse_addresses(self, addresses : list[str]) -> str:
        messages = [[
            {"role": "user", "content": self.prompt % {"address" : address}}
        ] for address in addresses]
        result = self.pipe(messages)
        responses = [self.output_parser(r[0]["generated_text"][1]["content"], original_address=addr) for r, addr in zip(result, addresses)]
        return responses

def extract_json_block(model_response : str):
    limit_chars = [('{', '}'), ('[', ']'), ('"', '"')]
    json_str = model_response
    parts = model_response.split("```")
    if len(parts) >= 2: # single code block or malformed code block
        json_str = parts[1]
    elif len(parts) >= 3:
        for part in parts:
            part = part.strip()
            if part.startswith("json") or any(part.startswith(lim[0]) and part.endswith(lim[1]) for lim in limit_chars):
                json_str = part
                break
    if json_str.startswith("json"):
        json_str = json_str[4:].strip()
    return json_str

In [181]:
import json

class JSONObjectParser:
    def __call__(self, response: str, original_address: str) -> dict:
        try:
            json_str = extract_json_block(response)
            obj = json.loads(json_str)
            result_builder = ParsedAddressResultBuilder(original_address)
            for k, v in obj.items():
                result_builder.add_part(k, v)
            result_builder.set_reserved("fullResponse", response)
            data = result_builder.build()
        except Exception as e:
            data = {"fullResponse": response, "error": str(e)}
        return data

class JSONTuplesParser:
    def __init__(self, ignore_other = True):
        self.ignore_other = ignore_other

    def __call__(self, response: str, original_address: str) -> dict:
        ignore_key = None
        if self.ignore_other:
            ignore_key = "Other"
            if isinstance(self.ignore_other, str):
                ignore_key = self.ignore_other
        try:
            json_str = extract_json_block(response)
            tuples = json.loads(json_str)
            result_builder = ParsedAddressResultBuilder(original_address)
            for part, ptype in tuples:
                if ptype != ignore_key:
                    result_builder.add_part(ptype, part)
            result_builder.set_reserved("fullResponse", response)
            data = result_builder.build()
        except Exception as e:
            data = {"fullResponse": response, "error": str(e)}
        return data

In [182]:
from collections import OrderedDict
import json
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv")
bzkopen_test = pd.read_csv("open_data/bzkopen_addresses_test.csv")

EXAMPLES = [
    ("Berlin, Alexanderplatz 1, 10178", 
     OrderedDict([("City" , "Berlin"), ("StreetName", "Alexanderplatz"), ("HouseNumber", "1")])),
    ("Braunschweig, Uferstr. 25", # From BZK open training set
     OrderedDict([("City", "Braunschweig"), ("StreetName", "Uferstr."), ("HouseNumber", "25")])),
    ("808 Westend Avenue, New York 25, N.Y.", # From BZK open training set
        OrderedDict([("StreetName", "Westend Avenue"), ("HouseNumber", "808"), 
        ("City", "New York"), ("State", "N.Y.")])),
]

PROMPTS = [
    "Segment addresses into their components.\n"
    "Output only a JSON object with the following keys: " + ", ".join(COLS_TO_PREDICT) + ". "
    "Do not include fields that cannot be determined and do not try to guess their values. "
    "For example, if the address is simply \"Berlin\" then the field \"Country\" should be null. "
    "Addresses will most times be written in german, meaning country and city names may be in "
    "german and the addresses "
    "may include german terms such as:\n"
    " - \"burg\" or \"stadt\" for city\n"
    " - \"straße\" or its abbreviation \"str.\" for street.\n"
    "These terms may occur as a suffix to another word. "
    "Consider the following examples:\n" +
    ''.join(f"Address: {adr}\n```json\n{json.dumps(ex)}\n```\n" for adr, ex in EXAMPLES) +
    "Now segment the following address:\n%(address)s",

    "Annotate address components with the respective types. "
    "Consider the component types: " + ", ".join(COLS_TO_PREDICT + ["Other"]) + ". "
    "Not all addresses will contain all component types and you must not guess the missing ones. "
    "Addresses will most times be written in german, meaning country and city names may be in "
    "german and the addresses "
    "may include german terms such as:\n"
    " - \"burg\" or \"stadt\" for city\n"
    " - \"straße\" or its abbreviation \"str.\" for street.\n"
    "These terms may occur as a suffix to another word.\n"
    "Output only a JSON list of [component, type] tuples. "
    "Consider the following examples:\n" +
    ''.join(f"Address: {adr}\n```json\n{json.dumps([(v,k) for k,v in ex.items()])}\n```\n" for adr, ex in EXAMPLES) +
    "Now annotate the following address:\n%(address)s",
]
for i, prompt in enumerate(PROMPTS):
    print(f"Prompt {i}:")
    print(prompt)
    print()

Prompt 0:
Segment addresses into their components.
Output only a JSON object with the following keys: HouseNumber, StreetName, City, State, Country. Do not include fields that cannot be determined and do not try to guess their values. For example, if the address is simply "Berlin" then the field "Country" should be null. Addresses will most times be written in german, meaning country and city names may be in german and the addresses may include german terms such as:
 - "burg" or "stadt" for city
 - "straße" or its abbreviation "str." for street.
These terms may occur as a suffix to another word. Consider the following examples:
Address: Berlin, Alexanderplatz 1, 10178
```json
{"City": "Berlin", "StreetName": "Alexanderplatz", "HouseNumber": "1"}
```
Address: Braunschweig, Uferstr. 25
```json
{"City": "Braunschweig", "StreetName": "Uferstr.", "HouseNumber": "25"}
```
Address: 808 Westend Avenue, New York 25, N.Y.
```json
{"StreetName": "Westend Avenue", "HouseNumber": "808", "City": "Ne

In [183]:

import time
from pathlib import Path

model_configs = [
    {
        "name" : "libpostal",
        "factory": lambda: LibpostalClient(),
        "cleanup": lambda client: client.close(),
    },
    {
        "name" : "libpostal-expanded",
        "factory": lambda: LibpostalClient(expand_before_parsing=True),
        "cleanup": lambda client: client.close(),
    },
    {
        "name" : "deepparse-bpemb",
        "factory": lambda: DeepParseParser(model_type="bpemb"),
    },
    {
        "name" : "deepparse-fasttext",
        "factory": lambda: DeepParseParser(model_type="fasttext"),
    },
    {
        "name" : "deepparse-bpemb-attention",
        "factory": lambda: DeepParseParser(model_type="bpemb", attention_mechanism=True),
    },
    {
        "name" : "deepparse-fasttext-attention",
        "factory": lambda: DeepParseParser(model_type="fasttext", attention_mechanism=True),
    },
    {
        "name" : "Llama-3.2-1B-Instruct-prompt0",
        "factory": lambda: LlamaAddressParsingModel(
            "meta-llama/Llama-3.2-1B-Instruct",
            PROMPTS[0],
            JSONObjectParser()
        ),
    },
    {
        "name" : "Llama-3.2-3B-Instruct-prompt0",
        "factory": lambda: LlamaAddressParsingModel(
            "meta-llama/Llama-3.2-3B-Instruct",
            PROMPTS[0],
            JSONObjectParser()
        ),
    },
    {
        "name" : "Llama-3.2-1B-Instruct-prompt1",
        "factory": lambda: LlamaAddressParsingModel(
            "meta-llama/Llama-3.2-1B-Instruct",
            PROMPTS[1],
            JSONTuplesParser()
        ),
    },
    {
        "name" : "Llama-3.2-3B-Instruct-prompt1",
        "factory": lambda: LlamaAddressParsingModel(
            "meta-llama/Llama-3.2-3B-Instruct",
            PROMPTS[1],
            JSONTuplesParser()
        ),
    },
]



def eval(dataset, configs, cols_to_predict, pred_cache_path=None):
    if pred_cache_path is not None:
        pred_cache_path = Path(pred_cache_path)
    if pred_cache_path is None or not pred_cache_path.exists():
        cached_preds = {}
    else:
        print(f"Loading cached predictions...")
        with open(pred_cache_path, "r") as f:
            cached_preds = json.load(f)
    
    preds_per_config = []
    model_results = []

    for config in configs:
        config_name = config["name"]
        if config_name in cached_preds:
            print(f"Using cached predictions for model {config_name}... To avoid this delete or rename the file {pred_cache_path} or delete the entry for {config_name} inside it.")
            preds = cached_preds[config_name]["preds"]
            deltatime = cached_preds[config_name]["deltatime"]
        else:
            print(f"Loading model {config_name}...")
            model = config["factory"]()
            print(f"Segmenting addresses...")
            start = time.monotonic()
            preds = model.parse_addresses(dataset["FullAddress"].tolist())
            deltatime = time.monotonic() - start
            if "cleanup" in config:
                print("Cleaning up model resources...")
                config["cleanup"](model)
            print("Parsing model responses...")
            if pred_cache_path is not None:
                cached_preds[config_name] = {
                    "preds": preds,
                    "deltatime": deltatime
                }
        preds_df = pd.DataFrame(preds)
        preds_per_config.append(preds_df)
        print("Computing metrics...")
        metrics = compare_preds(preds_df, dataset[cols_to_predict], target_columns=cols_to_predict)
        metrics["deltatime"] = deltatime
        metrics["rate"] = len(dataset) / metrics["deltatime"]
        metrics["error"] = preds_df["error"].notna().sum() if "error" in preds_df.columns else 0
        metrics["errorRate"] = metrics["error"] / len(dataset)
        preds_df["config_name"] = config_name
        preds_df["FullAddress"] = dataset["FullAddress"]

        model_results.append(metrics)
        print(f"Results for model {config_name}: {metrics}")

    if pred_cache_path is not None:
        with open(pred_cache_path, "w") as f:
            json.dump(cached_preds, f)

    preds_per_config_df = pd.concat(preds_per_config)
    default_cols = ["config_name", "FullAddress"] + cols_to_predict
    new_order = default_cols + [col for col in preds_per_config_df.columns if col not in default_cols]
    preds_per_config_df = preds_per_config_df[new_order]

    results_df = pd.DataFrame(model_results, index = [config["name"] for config in configs])
    return preds_per_config_df, results_df



preds_per_config, model_statistics = eval(bzkopen_val, model_configs, COLS_TO_PREDICT, pred_cache_path="cached_preds_val.json")

model_statistics

Loading model libpostal...
Attempting to start libpostal-server docker-compose service...
This may take a long time on first run since the docker image needs to be built.
Libpostal server is now available.
Segmenting addresses...
Cleaning up model resources...
Stopping auto-started libpostal-server docker container...
Parsing model responses...
Computing metrics...
Results for model libpostal: OrderedDict({'accuracy': 0.7651515151515151, 'precision': 0.650887573964497, 'recall': 0.44534412955465585, 'f1': 0.5288461538461539, 'accuracy_with_tol_1': 0.7712121212121212, 'accuracy_with_tol_2': 0.7772727272727272, 'accuracy_with_tol_3': 0.7833333333333333, 'accuracy_with_tol_4': 0.8136363636363636, 'average_levenshtein': 1.7454545454545454, 'average_similarity': 0.7890700578111242, 'average_levenshtein_match': 2.1215469613259668, 'average_similarity_match': 0.9590906780024714, 'no_match_rate': 0.17727272727272725, 'deltatime': 0.015000000013969839, 'rate': 8799.999991804361, 'error': 0, 'er

c:\Users\rpa\Documents\wiedergutmachung\bzk-post-processing\.conda\Lib\site-packages\deepparse\download_tools.py:92: UserWarning: The offline parameter is set to False, so if a new pre-trained `bpemb` model is available it will automatically be downloaded.
  warnings.warn(


Loading the embeddings model
Segmenting addresses...
Vectorizing the address
Parsing model responses...
Computing metrics...
Results for model deepparse-bpemb: OrderedDict({'accuracy': 0.4590909090909091, 'precision': 0.2938388625592417, 'recall': 0.25101214574898784, 'f1': 0.2707423580786026, 'accuracy_with_tol_1': 0.46060606060606063, 'accuracy_with_tol_2': 0.4727272727272727, 'accuracy_with_tol_3': 0.48484848484848486, 'accuracy_with_tol_4': 0.5, 'average_levenshtein': 3.2363636363636363, 'average_similarity': 0.512559877924037, 'average_levenshtein_match': 5.271820448877805, 'average_similarity_match': 0.8436147616704848, 'no_match_rate': 0.39242424242424245, 'deltatime': 1.6559999999590218, 'rate': 79.71014492950869, 'error': 0, 'errorRate': 0.0})
Loading model deepparse-fasttext...


c:\Users\rpa\Documents\wiedergutmachung\bzk-post-processing\.conda\Lib\site-packages\deepparse\download_tools.py:92: UserWarning: The offline parameter is set to False, so if a new pre-trained `fasttext` model is available it will automatically be downloaded.
  warnings.warn(


Loading the embeddings model
Segmenting addresses...
Vectorizing the address
Parsing model responses...
Computing metrics...
Results for model deepparse-fasttext: OrderedDict({'accuracy': 0.4984848484848485, 'precision': 0.36, 'recall': 0.32793522267206476, 'f1': 0.3432203389830509, 'accuracy_with_tol_1': 0.5060606060606061, 'accuracy_with_tol_2': 0.5151515151515151, 'accuracy_with_tol_3': 0.5257575757575758, 'accuracy_with_tol_4': 0.55, 'average_levenshtein': 2.5515151515151517, 'average_similarity': 0.5519472586955657, 'average_levenshtein_match': 3.9290780141843973, 'average_similarity_match': 0.8611943043476912, 'no_match_rate': 0.3590909090909091, 'deltatime': 0.35899999999674037, 'rate': 367.6880222874611, 'error': 0, 'errorRate': 0.0})
Loading model deepparse-bpemb-attention...


...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Segmenting addresses...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok

Parsing model responses...
Computing metrics...
Results for model Llama-3.2-3B-Instruct-prompt1: OrderedDict({'accuracy': 0.8303030303030303, 'precision': 0.7, 'recall': 0.6518218623481782, 'f1': 0.6750524109014675, 'accuracy_with_tol_1': 0.8363636363636363, 'accuracy_with_tol_2': 0.843939393939394, 'accuracy_with_tol_3': 0.8621212121212121, 'accuracy_with_tol_4': 0.8893939393939394, 'average_levenshtein': 1.0818181818181818, 'average_similarity': 0.8606487467179458, 'average_levenshtein_match': 1.2142857142857142, 'average_similarity_match': 0.9660343075405514, 'no_match_rate': 0.10909090909090913, 'deltatime': 262.75, 'rate': 0.5023786869647955, 'error': 1, 'errorRate': 0.007575757575757576})


,accuracy,precision,recall,f1,accuracy_with_tol_1,accuracy_with_tol_2,accuracy_with_tol_3,accuracy_with_tol_4,average_levenshtein,average_similarity,average_levenshtein_match,average_similarity_match,no_match_rate,deltatime,rate,error,errorRate
libpostal,0.765152,0.650888,0.445344,0.528846,0.771212,0.777273,0.783333,0.813636,1.745455,0.789070,2.121547,0.959091,0.177273,0.015,8799.999992,0,0.000000
libpostal-expanded,0.769697,0.578125,0.449393,0.505695,0.780303,0.803030,0.818182,0.848485,1.506061,0.818799,1.746924,0.949750,0.137879,0.015,8799.999992,0,0.000000
deepparse-bpemb,0.459091,0.293839,0.251012,0.270742,0.460606,0.472727,0.484848,0.500000,3.236364,0.512560,5.271820,0.843615,0.392424,1.656,79.710145,0,0.000000
deepparse-fasttext,0.498485,0.360000,0.327935,0.343220,0.506061,0.515152,0.525758,0.550000,2.551515,0.551947,3.929078,0.861194,0.359091,0.359,367.688022,0,0.000000
deepparse-bpemb-attention,0.363636,0.084270,0.060729,0.070588,0.374242,0.404545,0.427273,0.460606,3.384848,0.395304,6.955975,0.820442,0.518182,0.859,153.667055,0,0.000000
deepparse-fasttext-attention,0.322727,0.141700,0.141700,0.141700,0.330303,0.350000,0.369697,0.400000,4.056061,0.355795,8.909396,0.788003,0.548485,0.125,1056.000000,0,0.000000
Llama-3.2-1B-Instruct-prompt0,0.500000,0.338600,0.607287,0.434783,0.546970,0.566667,0.592424,0.695455,3.186364,0.529526,5.563492,0.924569,0.427273,117.485,1.123548,0,0.000000
Llama-3.2-3B-Instruct-prompt0,0.834848,0.675373,0.732794,0.702913,0.837879,0.842424,0.851515,0.886364,1.218182,0.858277,1.358108,0.956863,0.103030,256.391,0.514839,6,0.045455
Llama-3.2-1B-Instruct-prompt1,0.771212,0.590909,0.526316,0.556745,0.781818,0.798485,0.816667,0.840909,1.659091,0.809403,1.951872,0.952239,0.150000,142.859,0.923988,21,0.159091
Llama-3.2-3B-Instruct-prompt1,0.830303,0.700000,0.651822,0.675052,0.836364,0.843939,0.862121,0.889394,1.081818,0.860649,1.214286,0.966034,0.109091,262.750,0.502379,1,0.007576


In [184]:
preds_per_config

,config_name,FullAddress,HouseNumber,StreetName,City,State,Country,house,state_district,unit,...,District,StreetSuffix,error,County,Direction,Preposition,Kingdom/State,City/Other,City/Town,StreetName/Other
0,libpostal,"Regensburg, Königstr. 2/I",2/i,königstr.,NaN,NaN,NaN,regensburg,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,libpostal,Dortmund,NaN,NaN,dortmund,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,libpostal,Jöhlingen/Krs. Durlach/Baden.,NaN,NaN,NaN,NaN,NaN,jöhlingen/krs. durlach/baden.,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,libpostal,"8 Burlington Road, Manchester 20/England.",8___20/england.,burlington road manchester,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,libpostal,Leer/Ostfriesland,NaN,NaN,NaN,NaN,NaN,NaN,leer/ostfriesland,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,Llama-3.2-3B-Instruct-prompt1,Sosnowice/Polen,NaN,NaN,Sosnowice,NaN,Polen,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
128,Llama-3.2-3B-Instruct-prompt1,2114-79 St. Jackson Heights. N.Y. USA,2114-79,St.,NaN,N.Y.,USA,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
129,Llama-3.2-3B-Instruct-prompt1,Losone CSR,25,Losone,NaN,CSR,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
130,Llama-3.2-3B-Instruct-prompt1,Rum.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [185]:
default_cols = ["config_name", "FullAddress"] + COLS_TO_PREDICT
preds_vs_trues = preds_per_config[default_cols].merge(
    bzkopen_val[default_cols[1:]], on="FullAddress", suffixes=("_pred", "_true"), how="left")
preds_vs_trues = preds_vs_trues[["config_name", "FullAddress"] + [new_col for col in COLS_TO_PREDICT for new_col in [f"{col}_pred", f"{col}_true"]]] # Order the columns for readability
preds_vs_trues

,config_name,FullAddress,HouseNumber_pred,HouseNumber_true,StreetName_pred,StreetName_true,City_pred,City_true,State_pred,State_true,Country_pred,Country_true
0,libpostal,"Regensburg, Königstr. 2/I",2/i,2/I,königstr.,Königstr.,NaN,Regensburg,NaN,NaN,NaN,NaN
1,libpostal,Dortmund,NaN,NaN,NaN,NaN,dortmund,Dortmund,NaN,NaN,NaN,NaN
2,libpostal,Jöhlingen/Krs. Durlach/Baden.,NaN,NaN,NaN,NaN,NaN,Jöhlingen,NaN,Baden,NaN,NaN
3,libpostal,"8 Burlington Road, Manchester 20/England.",8___20/england.,8,burlington road manchester,Burlington Road,NaN,Manchester,NaN,NaN,NaN,England
4,libpostal,Leer/Ostfriesland,NaN,NaN,NaN,NaN,NaN,Leer,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1435,Llama-3.2-3B-Instruct-prompt1,Sosnowice/Polen,NaN,NaN,NaN,NaN,Sosnowice,Sosnowice,NaN,NaN,Polen,Polen
1436,Llama-3.2-3B-Instruct-prompt1,2114-79 St. Jackson Heights. N.Y. USA,2114-79,2114,St.,79 St.,NaN,N.Y.,N.Y.,NaN,USA,USA
1437,Llama-3.2-3B-Instruct-prompt1,Losone CSR,25,NaN,Losone,NaN,NaN,Losone,CSR,NaN,NaN,CSR
1438,Llama-3.2-3B-Instruct-prompt1,Rum.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rum.


In [186]:
_, house_street_city = eval(bzkopen_val, model_configs, ["HouseNumber", "StreetName", "City"], pred_cache_path="cached_preds_val.json")
house_street_city

Loading cached predictions...
Using cached predictions for model libpostal... To avoid this delete or rename the file cached_preds_val.json or delete the entry for libpostal inside it.
Computing metrics...
Results for model libpostal: OrderedDict({'accuracy': 0.6994949494949495, 'precision': 0.6442953020134228, 'recall': 0.4752475247524752, 'f1': 0.547008547008547, 'accuracy_with_tol_1': 0.7070707070707071, 'accuracy_with_tol_2': 0.7146464646464646, 'accuracy_with_tol_3': 0.7146464646464646, 'accuracy_with_tol_4': 0.7474747474747475, 'average_levenshtein': 2.457070707070707, 'average_similarity': 0.7370864599882372, 'average_levenshtein_match': 3.0987261146496814, 'average_similarity_match': 0.9295740068641463, 'no_match_rate': 0.20707070707070707, 'deltatime': 0.015000000013969839, 'rate': 8799.999991804361, 'error': 0, 'errorRate': 0.0})
Using cached predictions for model libpostal-expanded... To avoid this delete or rename the file cached_preds_val.json or delete the entry for libpo

,accuracy,precision,recall,f1,accuracy_with_tol_1,accuracy_with_tol_2,accuracy_with_tol_3,accuracy_with_tol_4,average_levenshtein,average_similarity,average_levenshtein_match,average_similarity_match,no_match_rate,deltatime,rate,error,errorRate
libpostal,0.699495,0.644295,0.475248,0.547009,0.707071,0.714646,0.714646,0.747475,2.457071,0.737086,3.098726,0.929574,0.207071,0.015,8799.999992,0,0.000000
libpostal-expanded,0.691919,0.562500,0.445545,0.497238,0.704545,0.729798,0.747475,0.785354,2.154040,0.765113,2.584848,0.918136,0.166667,0.015,8799.999992,0,0.000000
deepparse-bpemb,0.535354,0.293532,0.292079,0.292804,0.537879,0.558081,0.570707,0.583333,4.729798,0.622953,6.061489,0.798348,0.219697,1.656,79.710145,0,0.000000
deepparse-fasttext,0.593434,0.360976,0.366337,0.363636,0.606061,0.621212,0.636364,0.661616,3.583333,0.676810,4.379630,0.827212,0.181818,0.359,367.688022,0,0.000000
deepparse-bpemb-attention,0.467172,0.047619,0.024752,0.032573,0.484848,0.527778,0.547980,0.578283,4.237374,0.505506,6.712000,0.800721,0.368687,0.859,153.667055,0,0.000000
deepparse-fasttext-attention,0.378788,0.127072,0.113861,0.120104,0.391414,0.419192,0.444444,0.479798,5.361111,0.425026,9.520179,0.754755,0.436869,0.125,1056.000000,0,0.000000
Llama-3.2-1B-Instruct-prompt0,0.404040,0.380952,0.712871,0.496552,0.482323,0.510101,0.540404,0.656566,3.666667,0.450219,7.082927,0.869692,0.482323,117.485,1.123548,0,0.000000
Llama-3.2-3B-Instruct-prompt0,0.785354,0.672811,0.722772,0.696897,0.787879,0.795455,0.808081,0.845960,1.641414,0.821974,1.857143,0.930004,0.116162,256.391,0.514839,6,0.045455
Llama-3.2-1B-Instruct-prompt1,0.699495,0.577540,0.534653,0.555270,0.714646,0.729798,0.752525,0.782828,2.330808,0.759359,2.822630,0.919591,0.174242,142.859,0.923988,21,0.159091
Llama-3.2-3B-Instruct-prompt1,0.782828,0.705263,0.663366,0.683673,0.787879,0.792929,0.815657,0.848485,1.482323,0.824543,1.701449,0.946432,0.128788,262.750,0.502379,1,0.007576


In [187]:
metric = "f1"
metric_per_column = pd.DataFrame(index=model_statistics.index, columns=COLS_TO_PREDICT)
for col in COLS_TO_PREDICT:
    for config in model_statistics.index:
        preds = preds_per_config[preds_per_config["config_name"] == config]
        if col in preds.columns:
            metric_per_column.at[config, col] = compare_preds(preds, bzkopen_val, target_columns=[col])[metric]
        else:
            metric_per_column.at[config, col] = pd.NA
print(f"Per-column {metric} scores:")
metric_per_column

Per-column f1 scores:


,HouseNumber,StreetName,City,State,Country
libpostal,0.75,0.604651,0.432432,0.444444,0.428571
libpostal-expanded,0.7,0.238095,0.525253,0.333333,0.584615
deepparse-bpemb,0.704225,0.036036,0.289593,0.0,0.12
deepparse-fasttext,0.625,0.081633,0.393013,0.0,0.233333
deepparse-bpemb-attention,0.130435,0.021978,0.011765,0.0,0.176991
deepparse-fasttext-attention,0.392857,0.014184,0.11828,0.0,0.226415
Llama-3.2-1B-Instruct-prompt0,0.368098,0.285714,0.722892,0.131148,0.081633
Llama-3.2-3B-Instruct-prompt0,0.645833,0.681319,0.724138,0.555556,0.769231
Llama-3.2-1B-Instruct-prompt1,0.651163,0.404494,0.579439,0.4,0.603175
Llama-3.2-3B-Instruct-prompt1,0.697674,0.622222,0.703704,0.470588,0.676471


In [188]:
bzk_fields = bzkopen_val["field"].unique()
print(f"bzk_fields: {bzk_fields}")
metric_per_bzk_field = pd.DataFrame(index=model_statistics.index, columns=bzk_fields)

for field in bzk_fields:
    mask = bzkopen_val["field"] == field
    subset = bzkopen_val[mask]
    for config in model_statistics.index:
        preds = preds_per_config[preds_per_config["config_name"] == config]
        metric_per_bzk_field.at[config, field] = compare_preds(preds[mask], subset, target_columns=COLS_TO_PREDICT)[metric]
print(f"Per-BZK-field {metric} scores:")
metric_per_bzk_field

bzk_fields: <StringArray>
['ApplicantCurrentAddress',        'VictimBirthPlace',
     'ApplicantBirthPlace',        'VictimDeathPlace']
Length: 4, dtype: str
Per-BZK-field f1 scores:


,ApplicantCurrentAddress,VictimBirthPlace,ApplicantBirthPlace,VictimDeathPlace
libpostal,0.528,0.526316,0.542373,0.4
libpostal-expanded,0.416988,0.55814,0.640625,0.888889
deepparse-bpemb,0.220532,0.425532,0.304348,0.4
deepparse-fasttext,0.265683,0.666667,0.380282,0.363636
deepparse-bpemb-attention,0.0625,0.064516,0.09901,0.0
deepparse-fasttext-attention,0.14094,0.208333,0.130435,0.0
Llama-3.2-1B-Instruct-prompt0,0.553191,0.3125,0.337449,0.272727
Llama-3.2-3B-Instruct-prompt0,0.711409,0.6,0.717949,0.727273
Llama-3.2-1B-Instruct-prompt1,0.592857,0.409091,0.553846,0.307692
Llama-3.2-3B-Instruct-prompt1,0.704225,0.595745,0.652174,0.5
